# Bronze - CFPB Batch Backfill

## Setup

In [1]:
from pyspark.sql.functions import current_timestamp, input_file_name, lit, col

StatementMeta(, be8983fa-eb6a-41cf-aa5f-d500bf47de2a, 3, Finished, Available, Finished, False)

In [2]:
file_path = "Files/raw/complaints_2026-06-28_to_2026-08-21.csv"

start_date = "2026-06-28"
end_date = "2026-08-21"

batch_id = (
    f"cfpb_batch_{start_date.replace('-', '_')}"
    f"_to_{end_date.replace('-', '_')}"
)

bronze_table = "bronze.complaints"

StatementMeta(, be8983fa-eb6a-41cf-aa5f-d500bf47de2a, 4, Finished, Available, Finished, False)

## Read Batch File

In [3]:
df = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("inferSchema", "false")
    .option("multiLine", "true")
    .option("quote", '"')
    .option("escape", '"')
    .load(file_path)
    .where(
        (col("Date received") >= start_date)
        & (col("Date received") <= end_date)
    )
)

StatementMeta(, be8983fa-eb6a-41cf-aa5f-d500bf47de2a, 5, Finished, Available, Finished, False)

## Rename Columns

In [4]:
df_renamed = (
    df
    .withColumnRenamed("Date received", "date_received")
    .withColumnRenamed("Product", "product")
    .withColumnRenamed("Sub-product", "sub_product")
    .withColumnRenamed("Issue", "issue")
    .withColumnRenamed("Sub-issue", "sub_issue")
    .withColumnRenamed("Consumer complaint narrative", "consumer_complaint_narrative")
    .withColumnRenamed("Company public response", "company_public_response")
    .withColumnRenamed("Company", "company")
    .withColumnRenamed("State", "state")
    .withColumnRenamed("ZIP code", "zip_code")
    .withColumnRenamed("Tags", "tags")
    .withColumnRenamed("Consumer consent provided?", "consumer_consent_provided")
    .withColumnRenamed("Submitted via", "submitted_via")
    .withColumnRenamed("Date sent to company", "date_sent_to_company")
    .withColumnRenamed("Company response to consumer", "company_response_to_consumer")
    .withColumnRenamed("Timely response?", "timely_response")
    .withColumnRenamed("Consumer disputed?", "consumer_disputed")
    .withColumnRenamed("Complaint ID", "complaint_id")
)

StatementMeta(, be8983fa-eb6a-41cf-aa5f-d500bf47de2a, 6, Finished, Available, Finished, False)

## Add Bronze Metadata

In [5]:
df_bronze = (
    df_renamed
    .withColumn("ingestion_timestamp", current_timestamp())
    .withColumn("source_file", input_file_name())
    .withColumn("batch_id", lit(batch_id))
)

StatementMeta(, be8983fa-eb6a-41cf-aa5f-d500bf47de2a, 7, Finished, Available, Finished, False)

## Write Bronze Delta Table

In [6]:
(
    df_bronze.write
    .format("delta")
    .mode("append")
    .saveAsTable("bronze.complaints")
)

StatementMeta(, be8983fa-eb6a-41cf-aa5f-d500bf47de2a, 8, Finished, Available, Finished, False)